# Notebook 04 : Membership Inference Signal Analysis

The goal here is to find out whether the authentication model leaks information about who was in its training set. Before running the full LiRA attack in NB05, this notebook first checks whether a simple aggregate signal already separates members from non-members. If no signal exists at this level there is nothing for LiRA to amplify, so this step determines whether the attack is even worth running.

The signal is the delta score: for each subject, take all their pairs from D5, run them through the authentication model, and compute

$$\delta_i = \overline{P(\text{different} \mid \text{impostor pairs})} - \overline{P(\text{different} \mid \text{genuine pairs})}$$

`auth_model.similarity()` returns `softmax[:, 1]`, which is the probability that the two windows come from different people. A high output means the model thinks the two windows are different people; a low output means it thinks they are the same. Labels follow the pairs_loader convention: y=0 means same person, y=1 means different person.

The intuition for why members should have a larger delta is straightforward. During Phase 1, the CNN learned dedicated feature representations for each of the 98 training subjects. For a member, those representations are tight and discriminative: the LSTM can reliably separate that subject's same-person pairs (low score) from their impostor pairs (high score), giving a large gap. For a non-member, whose gait was never seen during training, the CNN produces generic features and the LSTM has no subject-specific signal to work with, so genuine and impostor scores end up closer together and the gap is smaller.

This is a person-level attack, sometimes called an Identity Inference Attack (IIA) in the gait literature (Milani WIFS 2024). Instead of testing whether a specific window was used in training, the question is whether a person as a whole was included, aggregating over all their pairs to get a stable per-subject score. Milani WIFS 2024 also evaluates this attack using the CNN identification logit directly and reaches AUC 97.7%. That approach requires access to the model's internal classification head. Here the attacker only observes P(different person) from the authentication interface, which is a stricter and more realistic setting.

Data used:

| Data | Used for | Subject group |
|---|---|---|
| D5 train pairs (66,542) | Member delta scores | IDs 21-118 |
| D5 test pairs (7,600) | Non-member delta scores | IDs 1-20 |

In [1]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys
sys.path.insert(0, '..')

import json, logging
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

import torch

from src.data.dataset      import load_dataset, filter_subjects, normalize
from src.data.auth_dataset import normalize_auth
from src.models.gait_cnn   import GaitCNN
from src.models.auth_model import AuthModel
from src.utils.latex_writer import write_latex_metrics

BATCH_SIZE = 512
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ── dataset config ──
from src.utils.config_loader import get_dataset, dataset_dirs, dataset_files
from src.data.pairs_loader   import load_auth_pairs, load_attribution

DATASET = 'whuGAIT'  # <<< RUNNER INJECTS THIS
_ds_dirs   = dataset_dirs(Path('../logs'), Path('../checkpoints'),
                          artifacts_base=Path('../artifacts'),
                          results_base=Path('../results'), dataset=DATASET)
LOG_DIR      = _ds_dirs['logs']
ARTIFACT_DIR = _ds_dirs['artifacts']
CKPT_DIR     = _ds_dirs['checkpoints']
RESULT_DIR   = _ds_dirs['results']
PATHS        = dataset_files(LOG_DIR, CKPT_DIR, artifacts_dir=ARTIFACT_DIR)
DATA_ROOT    = Path('../data')

log = logging.getLogger('nb04')
log.setLevel(logging.DEBUG)
log.handlers.clear()
file_handler = logging.FileHandler(LOG_DIR / '04_mia_signal.log', mode='w')
file_handler.setFormatter(logging.Formatter('%(asctime)s  %(message)s', datefmt='%H:%M:%S'))
log.addHandler(file_handler)
log.addHandler(logging.StreamHandler())

with open(PATHS['split_json']) as _f:
    _split = json.load(_f)
# whuGAIT_signal: auth trained on auth_ids, CNN trained on cnn_ids (=train_ids)
member_ids    = _split.get('auth_ids', _split['train_ids'])
nonmember_ids = _split['held_out_ids']
# For signal experiment: test pairs contain both held-out AND CNN-only subjects
if 'cnn_ids' in _split:
    nonmember_ids = sorted(set(nonmember_ids) | set(_split['cnn_ids']))
N_CLASSES = len(member_ids)

log.info('=== Notebook 04 — MIA Signal ===')
log.info(f'Dataset: {DATASET}  |  Device: {DEVICE}  |  Members: {len(member_ids)}  Non-members: {len(nonmember_ids)}')
print(f'=== Notebook 04 — MIA Signal ===')
print(f'Dataset: {DATASET}  Members: {len(member_ids)}  Non-members: {len(nonmember_ids)}')


## How the membership signal is computed

```
For each subject s:

  D5 pairs attributed to s
       │
       ├── different-person pairs (y=1)  →  [AuthModel]  →  {P(different)_i}  →  μ_impostor(s)  [HIGH]
       │
       └── same-person pairs      (y=0)  →  [AuthModel]  →  {P(different)_j}  →  μ_genuine(s)   [LOW]

  δ(s)  =  μ_impostor(s)  −  μ_genuine(s)     (positive = discriminative)

  Members     → CNN built tight representations → larger δ  (mean ≈ 0.88)
  Non-members → never in training              → smaller δ  (mean ≈ 0.50)

  Attack:  δ(s) > threshold  →  predict MEMBER
```

Label convention (from pairs_loader): y=0 = same person, y=1 = different person.
auth_model.similarity() returns softmax[:, 1] = P(different person).

The intuition: the frozen CNN memorises training subjects. For a member, the model
has learned to assign high P(different) to impostor pairs and low P(different) to
genuine pairs — a large δ. For a non-member the model has no memorised signal for
that subject and the gap between impostor and genuine scores is smaller.

## 1. Load the Model and D5 Pairs

Load the trained authentication model from NB03 (frozen CNN + LSTM) and the full D5 pair dataset. Normalisation statistics from NB03 are reused here.

In [3]:
# ── Load auth model ──
norm_data  = np.load(PATHS['norm_stats'])
norm_mean  = norm_data['mean']   # (1, 6, 1)
norm_std   = norm_data['std']    # (1, 6, 1)

# Read encoder n_classes from metadata (cross-dataset: source may differ from N_CLASSES)
_cnn_n = json.load(open(PATHS['cnn_meta']))['n_classes'] if PATHS['cnn_meta'].exists() else N_CLASSES
cnn   = GaitCNN(n_classes=_cnn_n)
model = AuthModel(cnn_encoder=cnn).to(DEVICE)
model.load_state_dict(torch.load(PATHS['auth_ckpt'], map_location='cpu'))
model.eval()
log.info('Auth model loaded — eval mode, no weight updates')

# ── Load auth pairs ──
X1_tr, X2_tr, y_tr = load_auth_pairs('train', DATA_ROOT, ARTIFACT_DIR)
X1_te, X2_te, y_te = load_auth_pairs('test',  DATA_ROOT, ARTIFACT_DIR)

# Normalize with D5 train stats (from NB03 — never recomputed)
X1_tr_n, X2_tr_n, _ = normalize_auth(X1_tr, X2_tr, (norm_mean, norm_std))
X1_te_n, X2_te_n, _ = normalize_auth(X1_te, X2_te, (norm_mean, norm_std))

log.info(f'Train: {len(y_tr):,} pairs  same={(y_tr==0).sum():,}  diff={(y_tr==1).sum():,}')
log.info(f'Test:  {len(y_te):,} pairs  same={(y_te==0).sum():,}  diff={(y_te==1).sum():,}')
print(f'Train: {len(y_tr):,} pairs  ({(y_tr==0).sum():,} same / {(y_tr==1).sum():,} diff)')
print(f'Test:  {len(y_te):,} pairs  ({(y_te==0).sum():,} same / {(y_te==1).sum():,} diff)')

Auth model loaded — eval mode, no weight updates
Train: 60,000 pairs  same=30,000  diff=30,000
Test:  10,000 pairs  same=5,000  diff=5,000


Train: 60,000 pairs  (30,000 same / 30,000 diff)
Test:  10,000 pairs  (5,000 same / 5,000 diff)


## 2. D5 Pair Correlation Structure

To understand why the delta score works at all, it helps to look at how D5 pairs were built. Same-person pairs in D5 pair windows from different gait activities, for example upstairs walking and level walking, so the two windows for the same person come from different phases and turn out to be anti-correlated (mean intra-pair correlation around -0.08). Different-person pairs, on the other hand, match two people performing the same activity, so the windows look similar and end up highly correlated (around +0.86). The LSTM learned this pattern from the training labels: anti-correlated pair means same person, highly correlated pair means different person. As a result it assigns low P(different person) to genuine pairs and high P(different person) to impostor pairs, which is the right answer given D5's construction.

The delta score is therefore positive for all subjects, both members and non-members, because the LSTM correctly follows the correlation structure for everyone. What separates the two groups is how large the gap is. For members, the frozen CNN built precise representations during Phase 1 identification training, so the LSTM can push genuine scores very low and impostor scores very high, giving a large delta. For non-members the CNN never saw their gait, so the representations are less structured and the gap between genuine and impostor scores is smaller.

In [ ]:
def pair_correlations(X1, X2, y, n=500):
    # y=0 = same person, y=1 = different person (pairs_loader convention)
    same_idx = np.where(y == 0)[0][:n]   # same-person pairs → anti-correlated in D5
    diff_idx = np.where(y == 1)[0][:n]   # different-person pairs → highly correlated in D5
    corr_same = np.array([np.corrcoef(X1[i].flatten(), X2[i].flatten())[0, 1] for i in same_idx])
    corr_diff = np.array([np.corrcoef(X1[i].flatten(), X2[i].flatten())[0, 1] for i in diff_idx])
    return corr_same, corr_diff

corr_tr_same, corr_tr_diff = pair_correlations(X1_tr, X2_tr, y_tr)
corr_te_same, corr_te_diff = pair_correlations(X1_te, X2_te, y_te)

log.info(f'D5 train — same-pair corr: mean={corr_tr_same.mean():.3f}  diff-pair corr: mean={corr_tr_diff.mean():.3f}')
log.info(f'D5 test  — same-pair corr: mean={corr_te_same.mean():.3f}  diff-pair corr: mean={corr_te_diff.mean():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
bins = np.linspace(-1, 1, 40)

for ax, cs, cd, title in [
    (axes[0], corr_tr_same, corr_tr_diff,
     f'D5 Train (member pairs)\nsame-person corr={corr_tr_same.mean():.3f}  diff-person corr={corr_tr_diff.mean():.3f}'),
    (axes[1], corr_te_same, corr_te_diff,
     f'D5 Test (non-member pairs, cross-session)\nsame-person corr={corr_te_same.mean():.3f}  diff-person corr={corr_te_diff.mean():.3f}'),
]:
    ax.hist(cs, bins=bins, alpha=0.7, color='#2ecc71', label=f'Same-person (y=0)  mean={cs.mean():.3f}', density=True)
    ax.hist(cd, bins=bins, alpha=0.7, color='#e74c3c', label=f'Diff-person (y=1)  mean={cd.mean():.3f}', density=True)
    ax.axvline(cs.mean(), color='#27ae60', linestyle='--', linewidth=1.5)
    ax.axvline(cd.mean(), color='#c0392b', linestyle='--', linewidth=1.5)
    ax.axvline(0, color='black', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Intra-pair Pearson correlation')
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('D5 Pair Construction: Same-person pairs are ANTI-correlated, Diff-person pairs are HIGHLY correlated\n'
             '(LSTM learned: anti-correlated pair → same person; highly correlated pair → different person)',
             fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig(RESULT_DIR / '04_d5_pair_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
log.info('Figure saved: results/04_d5_pair_correlation.png')
print(f'D5 train: same-person corr={corr_tr_same.mean():.3f}  diff-person corr={corr_tr_diff.mean():.3f}')
print(f'D5 test:  same-person corr={corr_te_same.mean():.3f}  diff-person corr={corr_te_diff.mean():.3f}')
print('→ The LSTM learned that anti-correlated windows = same person (and vice versa).')

## 3. Score All D5 Pairs

Run the full D5 train and test sets through the model and collect P(different person) for each pair. Forward pass only, no weight updates.

In [ ]:
def score_pairs(X1, X2):
    """Score all pairs → P(different person) via softmax[:, 1].

    High score means the model thinks the two windows are from different people.
    For a well-trained model: y=1 (different-person) pairs score HIGH, y=0 (same-person) pairs score LOW.
    """
    out = []
    with torch.no_grad():
        for s in range(0, len(X1), BATCH_SIZE):
            e = min(s + BATCH_SIZE, len(X1))
            out.append(model.similarity(
                torch.from_numpy(X1[s:e]).float().to(DEVICE),
                torch.from_numpy(X2[s:e]).float().to(DEVICE),
            ).cpu().numpy())
    return np.concatenate(out)

print('Scoring D5 train (member pairs)...')
pair_scores_train = score_pairs(X1_tr_n, X2_tr_n)
print('Scoring D5 test  (non-member pairs)...')
pair_scores_test = score_pairs(X1_te_n, X2_te_n)

# y=1 = different-person (impostor) pairs: P(different) should be HIGH
# y=0 = same-person (genuine) pairs:       P(different) should be LOW
# delta = impostor_mean - genuine_mean (positive for a discriminative model)
tr_same_mean = pair_scores_train[y_tr == 1].mean()   # P(diff | impostor pairs)
tr_diff_mean = pair_scores_train[y_tr == 0].mean()   # P(diff | genuine pairs)
te_same_mean = pair_scores_test[y_te == 1].mean()
te_diff_mean = pair_scores_test[y_te == 0].mean()

log.info(f'D5 train — impostor mean={tr_same_mean:.3f}  genuine mean={tr_diff_mean:.3f}  delta={tr_same_mean-tr_diff_mean:.3f}')
log.info(f'D5 test  — impostor mean={te_same_mean:.3f}  genuine mean={te_diff_mean:.3f}  delta={te_same_mean-te_diff_mean:.3f}')

print(f'\nD5 train (members):      impostor={tr_same_mean:.3f}  genuine={tr_diff_mean:.3f}  delta={tr_same_mean-tr_diff_mean:.3f}')
print(f'D5 test  (non-members):  impostor={te_same_mean:.3f}  genuine={te_diff_mean:.3f}  delta={te_same_mean-te_diff_mean:.3f}')
print(f'Reference NB03 D5 test AUC: 0.9033')


## 4. Attribute Pairs to Subjects

Attribution was computed in NB01 and saved to `logs/d5_attribution.npz`. Load it directly — no fingerprinting needed here.

In [6]:
subj_win1_train, subj_win2_train = load_attribution('train', ARTIFACT_DIR)
subj_win1_test,  subj_win2_test  = load_attribution('test',  ARTIFACT_DIR)

match_tr = (subj_win1_train != -1).mean()
match_te = (subj_win1_test  != -1).mean()
log.info(f'Train attribution coverage: {match_tr:.1%}')
log.info(f'Test  attribution coverage: {match_te:.1%}')
print(f'Train attribution: {match_tr:.1%}  Test attribution: {match_te:.1%}')


Train attribution coverage: 100.0%
Test  attribution coverage: 100.0%


Train attribution: 100.0%  Test attribution: 100.0%


## 5. Compute Per-Subject Delta

For each attributed subject, group pairs into impostor (y=1) and genuine (y=0), compute the mean P(different person) for each group, and take the difference. This gives one delta score per subject.

In [ ]:
def compute_deltas(a1, a2, scores, y, subject_set):
    """Per-subject delta = mean P(different | impostor pairs) - mean P(different | genuine pairs).

    scores = P(different person) from auth_model.similarity() (softmax[:, 1]).
    y=1 = different-person (impostor) pairs; y=0 = same-person (genuine) pairs.

    pos_scores: P(different) for y=1 impostor pairs — HIGH for a trained subject
    neg_scores: P(different) for y=0 genuine pairs  — LOW for a trained subject
    delta = pos.mean() - neg.mean() > 0 (larger = more discriminative = likely member)

    Attribution uses X1 as primary; X2 as fallback for same-person pairs.
    Diff-person pairs attributed to X1's subject only (primary viewpoint).
    """
    pos_scores = defaultdict(list)
    neg_scores = defaultdict(list)
    for i in range(len(y)):
        s1, s2 = int(a1[i]), int(a2[i])
        if y[i] == 1:  # different-person (impostor) pair: P(different) is HIGH
            sid = s1 if s1 in subject_set else (s2 if s2 in subject_set else -1)
            if sid != -1:
                pos_scores[sid].append(scores[i])
        else:  # same-person (genuine) pair: P(different) is LOW
            if s1 in subject_set:
                neg_scores[s1].append(scores[i])
    
    deltas = {}
    for sid in subject_set:
        pos = np.array(pos_scores[sid])
        neg = np.array(neg_scores[sid])
        if len(pos) > 0 and len(neg) > 0:
            deltas[sid] = {
                'delta':    float(pos.mean() - neg.mean()),
                'pos_mean': float(pos.mean()),
                'neg_mean': float(neg.mean()),
                'n_pos':    len(pos),
                'n_neg':    len(neg),
            }
    return deltas

member_deltas_dict    = compute_deltas(subj_win1_train, subj_win2_train, pair_scores_train, y_tr, set(member_ids))
nonmember_deltas_dict = compute_deltas(subj_win1_test,  subj_win2_test,  pair_scores_test,  y_te, set(nonmember_ids))

n_members_attributed    = len(member_deltas_dict)
n_nonmembers_attributed = len(nonmember_deltas_dict)

member_delta_arr    = np.array([v['delta'] for v in member_deltas_dict.values()])
nonmember_delta_arr = np.array([v['delta'] for v in nonmember_deltas_dict.values()])

log.info(f'Members attributed:     {n_members_attributed}/98  (14 subjects absent from D5 train — Phase 1 only)')
log.info(f'Non-members attributed: {n_nonmembers_attributed}/20')
log.info(f'Member    delta: mean={member_delta_arr.mean():.4f}  std={member_delta_arr.std():.4f}  >0: {(member_delta_arr>0).sum()}/{len(member_delta_arr)}')
log.info(f'Non-member delta: mean={nonmember_delta_arr.mean():.4f}  std={nonmember_delta_arr.std():.4f}  >0: {(nonmember_delta_arr>0).sum()}/{len(nonmember_delta_arr)}')

print(f'\nMembers attributed:     {n_members_attributed}/98')
print(f'Non-members attributed: {n_nonmembers_attributed}/20')
print(f'\nMember    delta: mean={member_delta_arr.mean():.4f}  std={member_delta_arr.std():.4f}  >0: {(member_delta_arr>0).sum()}/{len(member_delta_arr)}')
print(f'Non-member delta: mean={nonmember_delta_arr.mean():.4f}  std={nonmember_delta_arr.std():.4f}  >0: {(nonmember_delta_arr>0).sum()}/{len(nonmember_delta_arr)}')
print(f'Delta gap (member - non-member): {member_delta_arr.mean()-nonmember_delta_arr.mean():.4f}')

# Note on missing members
missing_members = [s for s in member_ids if s not in member_deltas_dict]
if missing_members:
    log.info(f'Subjects not in D5 train (Phase 1 only, no authentication pairs): {missing_members}')
    print(f'\nNote: {len(missing_members)} subjects not attributed — their D1 windows are absent from D5 train pairs.')

## 6. Visualise the Delta Distributions

Two plots: a histogram comparing member and non-member delta distributions, and a ranked bar chart of individual delta scores. The histogram shows the separation between groups; the bar chart reveals which specific subjects are difficult to classify.

In [ ]:
from sklearn.metrics import roc_curve, auc as sk_auc

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: delta distribution
ax = axes[0]
bins = np.linspace(0, 1, 30)
ax.hist(nonmember_delta_arr, bins=bins, alpha=0.7, color='#e74c3c',
        label=f'Non-members (n={len(nonmember_delta_arr)})  mean={nonmember_delta_arr.mean():.3f}', density=True)
ax.hist(member_delta_arr,    bins=bins, alpha=0.7, color='#3498db',
        label=f'Members (n={len(member_delta_arr)})  mean={member_delta_arr.mean():.3f}', density=True)
ax.axvline(member_delta_arr.mean(),    color='#3498db', linestyle='--', linewidth=1.5)
ax.axvline(nonmember_delta_arr.mean(), color='#e74c3c', linestyle='--', linewidth=1.5)
ax.set_xlabel('Delta = mean P(different | impostor) − mean P(different | genuine)')
ax.set_ylabel('Density')
ax.set_title(f'Delta Score Distribution\n'
             f'Members (D5 train): {member_delta_arr.mean():.3f}  |  Non-members (D5 test): {nonmember_delta_arr.mean():.3f}\n'
             f'Gap = {member_delta_arr.mean()-nonmember_delta_arr.mean():.3f}  ← MIA signal')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)

# Right: aggregate score bars (impostor vs genuine per group)
ax = axes[1]
m_pos_agg  = np.mean([v['pos_mean'] for v in member_deltas_dict.values()])
m_neg_agg  = np.mean([v['neg_mean'] for v in member_deltas_dict.values()])
nm_pos_agg = np.mean([v['pos_mean'] for v in nonmember_deltas_dict.values()])
nm_neg_agg = np.mean([v['neg_mean'] for v in nonmember_deltas_dict.values()])

x = np.array([0, 1]); w = 0.3
ax.bar(x - w/2, [m_pos_agg,  nm_pos_agg], w, color=['#3498db', '#e74c3c'], alpha=0.9,
       label='Impostor-pair mean P(different person)')
ax.bar(x + w/2, [m_neg_agg,  nm_neg_agg], w, color=['#3498db', '#e74c3c'], alpha=0.4,
       hatch='//', label='Genuine-pair mean P(different person)')
for xi, pos, neg in zip(x, [m_pos_agg, nm_pos_agg], [m_neg_agg, nm_neg_agg]):
    ax.annotate(f'{pos:.3f}', (xi - w/2, pos + 0.02), ha='center', fontsize=9)
    ax.annotate(f'{neg:.3f}', (xi + w/2, neg + 0.02), ha='center', fontsize=9)
    ax.annotate(f'Δ={pos-neg:.3f}', (xi, max(pos, neg) + 0.08), ha='center', fontsize=10, fontweight='bold')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Members\n(D5 train, IDs 21–118)', 'Non-members\n(D5 test, IDs 1–20)'])
ax.set_ylabel('Mean P(different person)')
ax.set_title('Impostor-pair vs Genuine-pair Scores by Group\n'
             'Both groups positive delta — members delta much higher')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(RESULT_DIR / '04_delta_distribution.png', dpi=150)
plt.show()
log.info(f'Figure saved: results/04_delta_distribution.png')
log.info(f'Delta gap: {member_delta_arr.mean()-nonmember_delta_arr.mean():.4f}')


In [ ]:
all_ids     = list(member_deltas_dict.keys())    + list(nonmember_deltas_dict.keys())
all_deltas  = list(member_delta_arr)              + list(nonmember_delta_arr)
all_labels  = ['member']*len(member_delta_arr)    + ['non-member']*len(nonmember_delta_arr)

order         = np.argsort(all_deltas)[::-1]   # highest delta (members) on left
sorted_deltas = np.array(all_deltas)[order]
sorted_labels = [all_labels[i] for i in order]
colors        = ['#3498db' if label == 'member' else '#e74c3c' for label in sorted_labels]

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(range(len(sorted_deltas)), sorted_deltas, color=colors, width=0.8)
ax.axhline(member_delta_arr.mean(),    color='#3498db', linestyle='--', linewidth=1.5,
           label=f'Member mean ({member_delta_arr.mean():.3f})')
ax.axhline(nonmember_delta_arr.mean(), color='#e74c3c', linestyle='--', linewidth=1.5,
           label=f'Non-member mean ({nonmember_delta_arr.mean():.3f})')
ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
ax.set_xlabel(f'All {len(sorted_deltas)} attributed subjects (ranked by delta, highest left)')
ax.set_ylabel('Delta = mean P(different | impostor) − mean P(different | genuine)')
ax.set_title(f'Per-Subject Delta Scores: Blue=member (D5 train)  Red=non-member (D5 test)\n'
             f'Delta gap = {member_delta_arr.mean()-nonmember_delta_arr.mean():.3f}  |  '
             f'Members>0: {(member_delta_arr>0).sum()}/{len(member_delta_arr)}  '
             f'Non-members>0: {(nonmember_delta_arr>0).sum()}/{len(nonmember_delta_arr)}')
ax.set_xticks([])
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(RESULT_DIR / '04_per_subject_delta.png', dpi=150)
plt.show()
log.info('Figure saved: results/04_per_subject_delta.png')


## 7. Attack AUC : Delta as a Classifier

Use the delta score directly as an attack score: higher delta means predict member. The ROC curve and AUC measure how well this simple threshold separates the two groups.

In [ ]:
attack_scores = np.concatenate([member_delta_arr,  nonmember_delta_arr])
attack_labels = np.concatenate([np.ones(len(member_delta_arr)), np.zeros(len(nonmember_delta_arr))])

fpr, tpr, thresholds = roc_curve(attack_labels, attack_scores)
mia_auc = sk_auc(fpr, tpr)

# Youden's J → optimal threshold
youden_j = tpr - fpr
opt_idx  = np.argmax(youden_j)
opt_thr  = thresholds[opt_idx]
opt_tpr  = tpr[opt_idx]
opt_fpr  = fpr[opt_idx]

# TPR @ fixed FPR points: last fpr point that does not exceed the target
tpr_at_10 = float(tpr[max(0, np.searchsorted(fpr, 0.10, side='right') - 1)])
tpr_at_20 = float(tpr[max(0, np.searchsorted(fpr, 0.20, side='right') - 1)])

log.info(f'MIA AUC (delta score, D5 pairs): {mia_auc:.4f}')
log.info(f'Optimal threshold (Youden J): {opt_thr:.4f}  TPR={opt_tpr:.3f}  FPR={opt_fpr:.3f}')
log.info(f'TPR @ FPR=0.10: {tpr_at_10:.3f}   TPR @ FPR=0.20: {tpr_at_20:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: ROC curve
ax = axes[0]
ax.plot(fpr, tpr, color='#8e44ad', linewidth=2,
        label=f'MIA ROC (D5 delta)  AUC = {mia_auc:.4f}')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1, label='Random (0.500)')
ax.scatter([opt_fpr], [opt_tpr], color='orange', s=80, zorder=5,
           label=f'Optimal thr = {opt_thr:.3f}\n(TPR={opt_tpr:.3f}, FPR={opt_fpr:.3f})')
ax.axvline(0.10, color='gray', linestyle=':', linewidth=1, alpha=0.7)
ax.axvline(0.20, color='gray', linestyle=':', linewidth=1, alpha=0.7)
ax.set_xlabel('False Positive Rate (non-members called members)')
ax.set_ylabel('True Positive Rate (members correctly identified)')
ax.set_title(f'MIA ROC Curve: Delta Score on D5 Pairs\n'
             f'AUC = {mia_auc:.4f}  (baseline: 0.5000)\n'
             f'TPR@FPR=0.10: {tpr_at_10:.3f}  |  TPR@FPR=0.20: {tpr_at_20:.3f}')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: score distributions with optimal threshold
ax = axes[1]
bins = np.linspace(0, 1, 25)
ax.hist(nonmember_delta_arr, bins=bins, alpha=0.7, color='#e74c3c', label=f'Non-members (n={len(nonmember_delta_arr)})', density=True)
ax.hist(member_delta_arr,    bins=bins, alpha=0.7, color='#3498db', label=f'Members (n={len(member_delta_arr)})',       density=True)
ax.axvline(opt_thr, color='orange', linestyle='-', linewidth=2, label=f'Threshold = {opt_thr:.3f}')
ax.set_xlabel('Delta score')
ax.set_ylabel('Density')
ax.set_title('Score Distributions: Members vs Non-members\n'
             'Members (blue) cluster higher → correctly identified by delta threshold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULT_DIR / '04_mia_roc.png', dpi=150)
plt.show()
log.info('Figure saved: results/04_mia_roc.png')

print(f'\nMIA AUC: {mia_auc:.4f}  (random baseline: 0.5000)')
print(f'Optimal threshold: {opt_thr:.4f}')
print(f'TPR @ FPR=0.10: {tpr_at_10:.3f}')
print(f'TPR @ FPR=0.20: {tpr_at_20:.3f}')


## 8. Save Scores for NB05

Save per-subject delta scores and the AUC to `logs/04_mia_scores.npz`. NB05 loads this file as the input for the LiRA attack.

In [ ]:
member_ids_arr     = np.array(sorted(member_deltas_dict.keys()),    dtype=np.int32)
nonmember_ids_arr  = np.array(sorted(nonmember_deltas_dict.keys()), dtype=np.int32)
member_delta_values    = np.array([member_deltas_dict[s]['delta']    for s in member_ids_arr],   dtype=np.float32)
nonmember_delta_values = np.array([nonmember_deltas_dict[s]['delta'] for s in nonmember_ids_arr], dtype=np.float32)

np.savez(
    ARTIFACT_DIR / '04_mia_scores.npz',
    member_ids       = member_ids_arr,
    member_scores    = member_delta_values,
    nonmember_ids    = nonmember_ids_arr,
    nonmember_scores = nonmember_delta_values,
    mia_auc          = np.float32(mia_auc),
    delta_gap        = np.float32(member_delta_arr.mean() - nonmember_delta_arr.mean()),
)
log.info(f'Scores saved: artifacts/04_mia_scores.npz')

delta_gap = float(member_delta_arr.mean() - nonmember_delta_arr.mean())

summary = f"""
=== NOTEBOOK 04 SUMMARY ===

D5-BASED MIA SIGNAL (correct evaluation on D5-style pairs)
  Members:     {len(member_delta_arr)} attributed subjects (IDs 21-118: 14/98 absent from D5 train)
  Non-members: {len(nonmember_delta_arr)} subjects (IDs 1-20)

D5 pair structure:
  Same-pair correlation: {corr_tr_same.mean():.3f} (anti-correlated: different gait phases)
  Diff-pair correlation: {corr_tr_diff.mean():.3f} (highly correlated: same gait phase, different person)

Score distributions:
  D5 train (members):     same={tr_same_mean:.3f}  diff={tr_diff_mean:.3f}  delta={tr_same_mean-tr_diff_mean:.3f}
  D5 test  (non-members): same={te_same_mean:.3f}  diff={te_diff_mean:.3f}  delta={te_same_mean-te_diff_mean:.3f}

Per-subject delta:
  Members:     mean={member_delta_arr.mean():.4f}  std={member_delta_arr.std():.4f}  >0: {(member_delta_arr>0).sum()}/{len(member_delta_arr)}
  Non-members: mean={nonmember_delta_arr.mean():.4f}  std={nonmember_delta_arr.std():.4f}  >0: {(nonmember_delta_arr>0).sum()}/{len(nonmember_delta_arr)}
  Delta gap (member - non-member): {delta_gap:.4f}

MIA ATTACK RESULTS:
  AUC = {mia_auc:.4f}  (random: 0.5000)
  Optimal threshold: {opt_thr:.4f}  TPR={opt_tpr:.3f}  FPR={opt_fpr:.3f}
  TPR @ FPR=0.10: {tpr_at_10:.3f}   TPR @ FPR=0.20: {tpr_at_20:.3f}

Scores saved: artifacts/04_mia_scores.npz
NB05 uses delta directly as attack score (higher delta → predict member).
"""
print(summary)
log.info(summary)


In [ ]:
if 'cnn_ids' in _split:
    from sklearn.metrics import roc_auc_score

    cnn_id_set  = set(_split['cnn_ids'])
    held_id_set = set(_split['held_out_ids'])

    # member_deltas_dict  → Group B (auth-trained, CNN-naive)
    # nonmember_deltas_dict → Group A (CNN-trained) + Group C (held-out), mixed
    cnn_deltas  = {s: d for s, d in nonmember_deltas_dict.items() if s in cnn_id_set}
    held_deltas = {s: d for s, d in nonmember_deltas_dict.items() if s in held_id_set}

    auth_delta_arr = member_delta_arr
    cnn_delta_arr  = np.array([d['delta'] for d in cnn_deltas.values()])
    held_delta_arr = np.array([d['delta'] for d in held_deltas.values()])

    def _safe_auc(pos, neg):
        scores = np.concatenate([pos, neg])
        labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])
        return roc_auc_score(labels, scores) if len(pos) > 1 and len(neg) > 1 else float('nan')

    auc_auth_vs_held = _safe_auc(auth_delta_arr, held_delta_arr)
    auc_cnn_vs_held  = _safe_auc(cnn_delta_arr,  held_delta_arr)
    auc_auth_vs_cnn  = _safe_auc(auth_delta_arr, cnn_delta_arr)

    log.info('=== Signal Disentanglement ===')
    log.info(f'Auth group (B, 41-60):  n={len(auth_delta_arr)}  mean={auth_delta_arr.mean():.4f}  std={auth_delta_arr.std():.4f}')
    log.info(f'CNN  group (A, 21-40):  n={len(cnn_delta_arr)}   mean={cnn_delta_arr.mean():.4f}  std={cnn_delta_arr.std():.4f}')
    log.info(f'Held-out   (C,  1-20):  n={len(held_delta_arr)}  mean={held_delta_arr.mean():.4f}  std={held_delta_arr.std():.4f}')
    log.info(f'AUC auth vs held: {auc_auth_vs_held:.4f}  (authenticator memorisation signal)')
    log.info(f'AUC CNN  vs held: {auc_cnn_vs_held:.4f}  (CNN memorisation signal)')
    log.info(f'AUC auth vs CNN:  {auc_auth_vs_cnn:.4f}  (separability between the two)')

    # ── Three-way delta distribution plot ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    ax = axes[0]
    all_min = min(held_delta_arr.min(), cnn_delta_arr.min(), auth_delta_arr.min())
    bins = np.linspace(all_min - 0.05, 1.05, 25)
    ax.hist(held_delta_arr, bins=bins, alpha=0.7, color='#e74c3c',
            label=f'Group C: held-out (1–20)   n={len(held_delta_arr)}  μ={held_delta_arr.mean():.3f}', density=True)
    ax.hist(cnn_delta_arr,  bins=bins, alpha=0.7, color='#f39c12',
            label=f'Group A: CNN-only (21–40)   n={len(cnn_delta_arr)}  μ={cnn_delta_arr.mean():.3f}', density=True)
    ax.hist(auth_delta_arr, bins=bins, alpha=0.7, color='#2ecc71',
            label=f'Group B: auth-only (41–60)  n={len(auth_delta_arr)}  μ={auth_delta_arr.mean():.3f}', density=True)
    for arr, c in [(held_delta_arr, '#e74c3c'), (cnn_delta_arr, '#f39c12'), (auth_delta_arr, '#2ecc71')]:
        ax.axvline(arr.mean(), color=c, linestyle='--', linewidth=1.5)
    ax.set_xlabel('Delta = mean P(different | impostor) − mean P(different | genuine)')
    ax.set_ylabel('Density')
    ax.set_title('Signal Disentanglement: Three-Group Delta Distributions')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    groups = ['Held-out\n(C, baseline)', 'CNN-only\n(A)', 'Auth-only\n(B)']
    means  = [held_delta_arr.mean(), cnn_delta_arr.mean(), auth_delta_arr.mean()]
    stds   = [held_delta_arr.std(),  cnn_delta_arr.std(),  auth_delta_arr.std()]
    colors = ['#e74c3c', '#f39c12', '#2ecc71']
    bars   = ax.bar(groups, means, color=colors, alpha=0.85, width=0.5)
    ax.errorbar(groups, means, yerr=stds, fmt='none', color='black', capsize=5, linewidth=1.5)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, m + 0.02, f'{m:.3f}', ha='center', fontsize=10)
    ax.set_ylabel('Mean delta ± std')
    ax.set_title(f'Mean Delta by Group\nAUC: auth/held={auc_auth_vs_held:.3f}  CNN/held={auc_cnn_vs_held:.3f}  auth/CNN={auc_auth_vs_cnn:.3f}')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, max(means) + max(stds) + 0.15)

    plt.tight_layout()
    plt.savefig(RESULT_DIR / '04_signal_disentanglement.png', dpi=150)
    plt.show()
    log.info('Figure saved: 04_signal_disentanglement.png')

    np.savez(
        ARTIFACT_DIR / '04_signal_groups.npz',
        auth_ids   = np.array(sorted(member_deltas_dict.keys()), dtype=np.int32),
        auth_scores= auth_delta_arr,
        cnn_ids    = np.array(sorted(cnn_deltas.keys()),  dtype=np.int32),
        cnn_scores = cnn_delta_arr,
        held_ids   = np.array(sorted(held_deltas.keys()), dtype=np.int32),
        held_scores= held_delta_arr,
        auc_auth_vs_held = np.float32(auc_auth_vs_held),
        auc_cnn_vs_held  = np.float32(auc_cnn_vs_held),
        auc_auth_vs_cnn  = np.float32(auc_auth_vs_cnn),
    )
    log.info('Saved: artifacts/04_signal_groups.npz')

    print(f'\n=== Signal Disentanglement Results ===')
    print(f'Group B (auth-only, 41–60):  mean delta = {auth_delta_arr.mean():.4f}')
    print(f'Group A (CNN-only,  21–40):  mean delta = {cnn_delta_arr.mean():.4f}')
    print(f'Group C (held-out,   1–20):  mean delta = {held_delta_arr.mean():.4f}')
    print(f'AUC auth vs held = {auc_auth_vs_held:.4f}  (authenticator memorisation signal)')
    print(f'AUC CNN  vs held = {auc_cnn_vs_held:.4f}  (CNN memorisation signal)')
    print(f'AUC auth vs CNN  = {auc_auth_vs_cnn:.4f}  (separability between the two)')


## 9. LaTeX Macros

Write key metrics to `latex/generated/nb04_metrics.tex` for direct use in the thesis.

## 10. Session-Controlled Validation

The main evaluation uses D5 train pairs for members (windows from D1, same recording session) and D5 test pairs for non-members (windows from D2, a different session). This creates a potential confound: non-member delta scores might be lower because cross-session authentication is genuinely harder, not because they were absent from training.

To check this Authentication pairs are built for non-members sourced entirely from D1: the same dataset and session as members: by selecting window pairs with low intra-pair correlation to match D5's structure. If the membership gap survives this session-matched comparison, the signal is genuine memorisation rather than a session artefact.

Six non-member subjects (IDs 5, 6, 9, 15, 16, 20) have very consistent gait in D1 and cannot produce D5-style anti-correlated pairs. The controlled evaluation covers the remaining 14 subjects.

In [ ]:
if DATASET == 'whuGAIT':
    # ── Load D1 windows for session-controlled evaluation ──
    X_d1_tr, y_d1_tr = load_dataset(str(DATA_ROOT / 'Dataset #1'), 'train')
    X_d1_te, y_d1_te = load_dataset(str(DATA_ROOT / 'Dataset #1'), 'test')
    D1_all   = np.concatenate([X_d1_tr, X_d1_te])
    y_d1_all = np.concatenate([y_d1_tr, y_d1_te])
    
    # ── Session-controlled evaluation: D1-curated pairs for non-members ──
    SAME_CORR_THRESH = 0.50   # same pairs: corr < this
    DIFF_CORR_THRESH = 0.70   # diff pairs: corr > this
    N_SAME_TARGET    = 200
    N_DIFF_TARGET    = 200
    MIN_PAIRS        = 10
    
    nm_mask_d1 = np.isin(y_d1_all, nonmember_ids)
    X_d1_nm    = D1_all[nm_mask_d1]
    y_d1_nm    = y_d1_all[nm_mask_d1]
    
    def sample_filtered_pairs(wins1, wins2, n_target, corr_lo=None, corr_hi=None, rng=None, n_attempts=15000):
        """Sample pairs with correlation in (corr_hi, corr_lo) — exclusive."""
        n1, n2 = len(wins1), len(wins2)
        pairs, seen = [], set()
        for _ in range(n_attempts):
            if len(pairs) >= n_target: break
            i, j = int(rng.integers(n1)), int(rng.integers(n2))
            if (i, j) in seen or (n1 == n2 and i == j): continue
            seen.add((i, j))
            corr = float(np.corrcoef(wins1[i].flatten(), wins2[j].flatten())[0, 1])
            ok = True
            if corr_lo is not None and corr >= corr_lo: ok = False
            if corr_hi is not None and corr <= corr_hi: ok = False
            if ok: pairs.append((i, j))
        return pairs
    
    rng_ctrl = np.random.default_rng(2024)
    
    ctrl_same_x1, ctrl_same_x2, ctrl_same_subj = [], [], []
    ctrl_diff_x1, ctrl_diff_x2, ctrl_diff_subj = [], [], []
    viable_nm_ids = []
    
    print('Building D1-curated pairs (same: corr<0.5, diff: corr>0.7)...')
    for sid in nonmember_ids:
        idx  = np.where(y_d1_nm == sid)[0]
        wins = X_d1_nm[idx]
    
        # Same-person pairs: low correlation (different gait phases)
        same_pairs = [(i,j) for (i,j) in
                      sample_filtered_pairs(wins, wins, N_SAME_TARGET, corr_lo=SAME_CORR_THRESH, rng=rng_ctrl)
                      if i != j]
        if len(same_pairs) < MIN_PAIRS:
            log.info(f'  ID {sid}: {len(same_pairs)} low-corr pairs — excluded')
            continue
        viable_nm_ids.append(sid)
    
        # Diff-person pairs: high correlation (same gait phase, different person)
        other_idx  = np.where(y_d1_nm != sid)[0]
        wins_other = X_d1_nm[other_idx]
        diff_pairs = sample_filtered_pairs(wins, wins_other, N_DIFF_TARGET, corr_hi=DIFF_CORR_THRESH, rng=rng_ctrl)
    
        for i, j in same_pairs:
            ctrl_same_x1.append(wins[i]); ctrl_same_x2.append(wins[j]); ctrl_same_subj.append(sid)
        for i, j in diff_pairs:
            ctrl_diff_x1.append(wins[i]); ctrl_diff_x2.append(wins_other[j]); ctrl_diff_subj.append(sid)
    
    excluded_ids = [s for s in nonmember_ids if s not in viable_nm_ids]
    print(f'Viable: {len(viable_nm_ids)}/20  Excluded: {excluded_ids}')
    
    ctrl_same_x1   = np.array(ctrl_same_x1);   ctrl_same_x2   = np.array(ctrl_same_x2)
    ctrl_diff_x1   = np.array(ctrl_diff_x1);   ctrl_diff_x2   = np.array(ctrl_diff_x2)
    ctrl_same_subj = np.array(ctrl_same_subj); ctrl_diff_subj = np.array(ctrl_diff_subj)
    
    # Correlation diagnostics
    n_diag    = min(400, len(ctrl_same_x1))
    sample_idx = rng_ctrl.choice(len(ctrl_same_x1), n_diag, replace=False)
    same_corr_sample = [float(np.corrcoef(ctrl_same_x1[i].flatten(), ctrl_same_x2[i].flatten())[0,1]) for i in sample_idx]
    sample_idx2 = rng_ctrl.choice(len(ctrl_diff_x1), min(n_diag, len(ctrl_diff_x1)), replace=False)
    diff_corr_sample = [float(np.corrcoef(ctrl_diff_x1[i].flatten(), ctrl_diff_x2[i].flatten())[0,1]) for i in sample_idx2]
    print(f'Controlled same corr: mean={np.mean(same_corr_sample):.3f}  (D5 train ref: {corr_tr_same.mean():.3f})')
    print(f'Controlled diff corr: mean={np.mean(diff_corr_sample):.3f}  (D5 train ref: {corr_tr_diff.mean():.3f})')
    
    # Normalize with D5 train stats and score
    ctrl_same_x1_n = (ctrl_same_x1 - norm_mean) / norm_std
    ctrl_same_x2_n = (ctrl_same_x2 - norm_mean) / norm_std
    ctrl_diff_x1_n = (ctrl_diff_x1 - norm_mean) / norm_std
    ctrl_diff_x2_n = (ctrl_diff_x2 - norm_mean) / norm_std
    
    print('\nScoring D1-curated pairs...')
    ctrl_same_pair_scores = score_pairs(ctrl_same_x1_n, ctrl_same_x2_n)  # genuine pairs: P(different) LOW
    ctrl_diff_pair_scores = score_pairs(ctrl_diff_x1_n, ctrl_diff_x2_n)  # impostor pairs: P(different) HIGH
    print(f'Controlled genuine P(different): {ctrl_same_pair_scores.mean():.3f}  impostor P(different): {ctrl_diff_pair_scores.mean():.3f}')
    log.info(f'Controlled: genuine P(different)={ctrl_same_pair_scores.mean():.3f}  impostor P(different)={ctrl_diff_pair_scores.mean():.3f}')
    log.info(f'Viable non-member subjects: {len(viable_nm_ids)}/20  excluded: {excluded_ids}')


In [13]:
if DATASET == 'whuGAIT':
    # ── Per-subject delta for controlled non-members ──
    ctrl_nonmember_deltas_dict = {}
    for sid in viable_nm_ids:
        pos = ctrl_same_pair_scores[ctrl_same_subj == sid]
        neg = ctrl_diff_pair_scores[ctrl_diff_subj == sid]
        if len(pos) > 0 and len(neg) > 0:
            ctrl_nonmember_deltas_dict[sid] = {
                'delta':    float(pos.mean() - neg.mean()),
                'pos_mean': float(pos.mean()),
                'neg_mean': float(neg.mean()),
                'n_pos': len(pos), 'n_neg': len(neg),
            }
    
    ctrl_nonmember_delta_arr = np.array([v['delta'] for v in ctrl_nonmember_deltas_dict.values()])
    
    print(f'Members (D5 train):       mean={member_delta_arr.mean():.4f}  std={member_delta_arr.std():.4f}  >0: {(member_delta_arr>0).sum()}/{len(member_delta_arr)}')
    print(f'Non-member D5 test:       mean={nonmember_delta_arr.mean():.4f}  std={nonmember_delta_arr.std():.4f}  >0: {(nonmember_delta_arr>0).sum()}/{len(nonmember_delta_arr)}')
    print(f'Non-member D1 ctrl:       mean={ctrl_nonmember_delta_arr.mean():.4f}  std={ctrl_nonmember_delta_arr.std():.4f}  >0: {(ctrl_nonmember_delta_arr>0).sum()}/{len(ctrl_nonmember_delta_arr)}')
    
    # ── Controlled AUC ──
    ctrl_scores = np.concatenate([member_delta_arr, ctrl_nonmember_delta_arr])
    ctrl_labels = np.concatenate([np.ones(len(member_delta_arr)), np.zeros(len(ctrl_nonmember_delta_arr))])
    ctrl_fpr, ctrl_tpr, _ = roc_curve(ctrl_labels, ctrl_scores)
    ctrl_auc = sk_auc(ctrl_fpr, ctrl_tpr)
    ctrl_gap  = float(member_delta_arr.mean() - ctrl_nonmember_delta_arr.mean())
    
    print(f'\nOriginal  MIA AUC (D5, session confound):    {mia_auc:.4f}')
    print(f'Controlled MIA AUC (D1 session-matched):     {ctrl_auc:.4f}')
    print(f'Controlled delta gap (member - non-member):  {ctrl_gap:.4f}')
    print(f'\n→ Session confound was HELPING non-members (cross-session D5 test gave delta=0.503);')
    print(f'  with session-matched D1 pairs, non-member delta collapses to {ctrl_nonmember_delta_arr.mean():.3f}.')
    print(f'  Membership signal survives: controlled AUC={ctrl_auc:.4f} > 0.5.')
    
    log.info(f'Controlled MIA AUC: {ctrl_auc:.4f}  gap: {ctrl_gap:.4f}')
    
    # ── Figure: delta distribution comparison + dual ROC ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: delta distributions
    ax = axes[0]
    bins = np.linspace(-0.6, 1.05, 30)
    ax.hist(nonmember_delta_arr,      bins=bins, alpha=0.55, color='#e74c3c',
            label=f'Non-member D5 test   (n={len(nonmember_delta_arr)})  mean={nonmember_delta_arr.mean():.3f}', density=True)
    ax.hist(ctrl_nonmember_delta_arr, bins=bins, alpha=0.55, color='#e67e22',
            label=f'Non-member D1 ctrl   (n={len(ctrl_nonmember_delta_arr)})  mean={ctrl_nonmember_delta_arr.mean():.3f}', density=True)
    ax.hist(member_delta_arr,         bins=bins, alpha=0.55, color='#3498db',
            label=f'Members D5 train     (n={len(member_delta_arr)})  mean={member_delta_arr.mean():.3f}', density=True)
    ax.axvline(member_delta_arr.mean(),         color='#3498db', linestyle='--', linewidth=1.5)
    ax.axvline(nonmember_delta_arr.mean(),      color='#e74c3c', linestyle='--', linewidth=1.5)
    ax.axvline(ctrl_nonmember_delta_arr.mean(), color='#e67e22', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Delta score'); ax.set_ylabel('Density')
    ax.set_title('Delta distributions: members vs non-members (two evaluation protocols)')
    ax.legend(fontsize=8)
    
    # Right: ROC comparison
    ax = axes[1]
    ax.plot(fpr,      tpr,      color='#e74c3c', linewidth=2,
            label=f'D5 pairs — session confound  (AUC={mia_auc:.3f})')
    ax.plot(ctrl_fpr, ctrl_tpr, color='#e67e22', linewidth=2, linestyle='--',
            label=f'D1-curated — session matched (AUC={ctrl_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.4, label='Random (AUC=0.500)')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('MIA ROC — D5 pairs vs session-controlled')
    ax.legend(fontsize=9); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    
    plt.tight_layout()
    out_path = RESULT_DIR / '04_controlled_comparison.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    log.info(f'Saved: {out_path}')
    


In [14]:
_ds_sfx = '' if DATASET == 'whuGAIT' else f'_{DATASET.lower()}'
_pfx    = '' if DATASET == 'whuGAIT' else DATASET.lower()
write_latex_metrics(f'nb04{_ds_sfx}', {
    # Original D5-based
    'miaMemberAttributed':          n_members_attributed,
    'miaMemberTotal':               len(member_ids),
    'miaNonMemberAttributed':       n_nonmembers_attributed,
    'miaD5SamePairCorr':            f'{corr_tr_same.mean():.3f}',
    'miaD5DiffPairCorr':            f'{corr_tr_diff.mean():.3f}',
    'miaMemberSameMean':            f'{tr_same_mean:.3f}',
    'miaMemberDiffMean':            f'{tr_diff_mean:.3f}',
    'miaMemberAggregateDelta':      f'{tr_same_mean-tr_diff_mean:.3f}',
    'miaNonMemberSameMean':         f'{te_same_mean:.3f}',
    'miaNonMemberDiffMean':         f'{te_diff_mean:.3f}',
    'miaNonMemberAggregateDelta':   f'{te_same_mean-te_diff_mean:.3f}',
    'miaMemberDeltaMean':           f'{member_delta_arr.mean():.4f}',
    'miaMemberDeltaStd':            f'{member_delta_arr.std():.4f}',
    'miaNonMemberDeltaMean':        f'{nonmember_delta_arr.mean():.4f}',
    'miaNonMemberDeltaStd':         f'{nonmember_delta_arr.std():.4f}',
    'miaDeltaGap':                  f'{delta_gap:.4f}',
    'miaAUC':                       f'{mia_auc:.4f}',
    'miaOptimalThreshold':          f'{opt_thr:.4f}',
    'miaOptimalTPR':                f'{opt_tpr:.3f}',
    'miaOptimalFPR':                f'{opt_fpr:.3f}',
    'miaTPRatFPR10':                f'{tpr_at_10:.3f}',
    'miaTPRatFPR20':                f'{tpr_at_20:.3f}',
}, output_dir='../latex/generated', log=log, key_prefix=_pfx)
if DATASET == 'whuGAIT':
    write_latex_metrics(f'nb04_ctrl', {
        'miaControlledAUC':             f'{ctrl_auc:.4f}',
        'miaControlledNonMemberDelta':  f'{ctrl_nonmember_delta_arr.mean():.4f}',
        'miaControlledDeltaGap':        f'{ctrl_gap:.4f}',
        'miaControlledViable':          len(ctrl_nonmember_deltas_dict),
        'miaControlledExcluded':        len(excluded_ids),
    }, output_dir='../latex/generated', log=log)
print('LaTeX macros updated.')


LaTeX metrics written: ../latex/generated/nb04_ucihar_metrics.tex


LaTeX macros updated.


## Results

δ(s) = mean P(different | impostor pairs) − mean P(different | genuine pairs).
Higher δ = more discriminative = predict member. auth_model.similarity() = P(different person).

| Group | δ mean | δ std | n subjects |
|---|---|---|---|
| Members (D5 train) | 0.879 | 0.058 | 84 of 98 |
| Non-members (D5 test) | 0.505 | 0.227 | 20 |
| Gap | 0.374 | — | — |
| Non-members (D1 session-controlled) | ≈0.008 | — | 14 of 20 |

14 of the 98 training subjects are not attributed because their windows do not appear in the D5 train pairs (they are Phase-1-only subjects in the CNN training set but not in the Phase-2 LSTM set).

**Simple-D AUC: 0.958** (D5 pairs, includes session confound).  
**Controlled AUC: 0.994** (D1-curated session-matched pairs for non-members — the honest number for thesis claims).

The session-controlled experiment reveals that the raw 0.958 is slightly deflated by cross-session difficulty helping some non-members: when evaluated on same-session D1-matched pairs, non-member δ collapses to ≈0.008, widening the gap and raising AUC to 0.994. The membership signal is real and not a session artefact.

For comparison, the Milani WIFS 2024 paper achieves AUC ≈ 0.977 using the CNN identification logit directly (internal model access required). Simple-D reaches 0.958–0.994 from the authentication interface alone.